# 5 — Report and resolve exact Results

> **Lesson focus**
>
> **Learn:** aggregate explicit Results and reload an exact request
> after restart. **Run:** build a report, then resolve the same quantity
> request. **Inspect:** durable request/Result identity rather than
> “latest.” **Status:** Human-`ACCEPTED` Full V1; stabilization in
> progress.

## Report only Results you already hold

The report specification names exact `AnalysisResult` inputs. It does
not solve, search a workspace for “latest,” or invent an acceptance
decision.

In [ ]:
from fixtures.primitive_resonator import build_primitive_resonator
from scnsim import (
    CircuitRun,
    DiagonalRootSpec,
    DirectSolveSpec,
    ReductionPipeline,
    ReportSpec,
    units as u,
)

workspace = "workspaces/primitive-course"
fixture = build_primitive_resonator()
run = CircuitRun(
    plan=fixture.plan,
    workspace=workspace,
)
direct_spec = DirectSolveSpec(
    frequencies=[5.5, 6.0, 6.5, 7.0] * u.GHz,
)
direct = run.solve(run.original, direct_spec)

quantity_view = run.original.reduce(
    ReductionPipeline().retain(fixture.resonator_node)
)
root_spec = DiagonalRootSpec(
    coordinate=fixture.resonator_node,
    root_hint=6.0 * u.GHz,
)
root = run.evaluate(quantity_view, root_spec)
report = run.build_report(ReportSpec(inputs=(direct, root)))
report.show()

> **Optional: preserve topology iterations**
>
> The course keeps its main path on the default replaceable workspace.
> If you want topology history, use a separate path and opt in
> deliberately:
>
> ``` python
> history_run = CircuitRun(
>     plan=fixture.plan,
>     workspace="workspaces/primitive-course-history",
>     versioned=True,
> )
> ```
>
> Distinct Plans then live under `iteration01`, `iteration02`, and so
> on; the same Plan returns to its existing iteration. Upgrading an
> existing unversioned workspace is supported, but switching that parent
> back to `versioned=False` is forbidden. Use another workspace instead.

## Reconstruct the exact request after restart

A new Run may resolve the same Plan, Ref, Spec, parameters, and
workspace identity. `resolve()` never executes and never falls back to
another receipt.

In [ ]:
from fixtures.primitive_resonator import build_primitive_resonator
from scnsim import CircuitRun, DiagonalRootSpec, ReductionPipeline, units as u

workspace = "workspaces/primitive-course"
fixture_after_restart = build_primitive_resonator()
run_after_restart = CircuitRun(
    plan=fixture_after_restart.plan,
    workspace=workspace,
)
quantity_view_after_restart = run_after_restart.original.reduce(
    ReductionPipeline().retain(fixture_after_restart.resonator_node)
)
root_spec_after_restart = DiagonalRootSpec(
    coordinate=fixture_after_restart.resonator_node,
    root_hint=6.0 * u.GHz,
)
resolved = run_after_restart.resolve(
    quantity_view_after_restart,
    root_spec_after_restart,
)
resolved.show()

A different Spec, ParameterSet, Plan, or Ref is a different request. A
different workspace is a different evidence location and fails closed if
that same exact request receipt is absent. `resolve()` never substitutes
the most recent compatible-looking Result.

[Previous](04_optimize_primitive.qmd) · [Course
map](../../docs/index.qmd) · [Next: create a custom
Library](06_create_library.qmd) · [Concept: exact Result
ownership](../../docs/concepts/requests-execution-and-result-ownership.qmd#run-request-result-ownership)